# 04. 미기후 — NDVI (식생/냉각축)

여름 NDVI 합성을 격자에 결합한다. NDVI는 열스트레스 지수의 **냉각축**(식생·증발산)이자,
8/5 구름 갭필의 핵심 예측변수. (SVF·UTCI는 외부데이터 수집 후 별도 노트북)

- `ndvi`: 여름 합성 NDVI (−1~1, 높을수록 식생 많음)

산출물: `data/processed/grid_features.geojson`(ndvi 컬럼 추가), `output/figures/fig04_ndvi_grid.png`

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, geopandas as gpd
from src.config import (DATA_PROCESSED, OUT_FIGURES, CRS_LANDSAT, CRS_SUBMIT,
                        LANDSAT_SUMMER, LANDSAT_PEAK, landsat_file)
from src.raster.landsat import ndvi_composite
from src.raster.geo import read_geotransform, pixel_center_coords

grid = gpd.read_file(DATA_PROCESSED / "grid_features.geojson").to_crs(CRS_LANDSAT)
ndvi = ndvi_composite(LANDSAT_SUMMER)
round(float(np.nanmean(ndvi)), 3), round(float(np.nanmin(ndvi)), 2), round(float(np.nanmax(ndvi)), 2)

(0.595, -0.52, 0.94)

In [2]:
# 픽셀 중심점 → 격자 평균 NDVI
gt = read_geotransform(str(landsat_file(LANDSAT_PEAK, "ST_B10")))
xs, ys = pixel_center_coords(gt["gt"], gt["width"], gt["height"])
v = ~np.isnan(ndvi)
pts = gpd.GeoDataFrame({"ndvi": ndvi[v]}, geometry=gpd.points_from_xy(xs[v], ys[v]), crs=CRS_LANDSAT)
j = gpd.sjoin(pts, grid[["grid_id", "geometry"]], predicate="within", how="inner")
agg = j.groupby("grid_id")["ndvi"].mean().reset_index()
grid = grid.drop(columns=["ndvi"], errors="ignore").merge(agg, on="grid_id", how="left")
print(f"NDVI 있는 격자 {grid['ndvi'].notna().sum():,} / {len(grid):,}")

NDVI 있는 격자 54,029 / 54,029


In [3]:
# 지구별 NDVI (그린필드 vs 도시) + LST와의 상관
print(grid.groupby("zoneName")["ndvi"].mean().round(3))
print("\nNDVI-LST(8월지수) 상관:", round(grid[["ndvi", "lst_z"]].corr().iloc[0, 1], 3))

zoneName
고양창릉 공공주택지구      0.525
남양주왕숙 공공주택지구     0.507
성남분당             0.505
성남판교지구 택지개발사업    0.547
Name: ndvi, dtype: float32

NDVI-LST(8월지수) 상관: -0.682


In [4]:
# 저장 (ndvi 컬럼 추가) + 지도
grid.to_crs(CRS_SUBMIT).to_file(DATA_PROCESSED / "grid_features.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

import matplotlib.pyplot as plt
from src.viz.fonts import setup_font
setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
ax = grid.plot(column="ndvi", cmap="YlGn", figsize=(9, 9), linewidth=0, legend=True,
               legend_kwds={"shrink": 0.6}, vmin=0, vmax=0.6, missing_kwds={"color": "#eeeeee"})
ax.set_axis_off(); ax.set_title("여름 합성 NDVI (식생)")
plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig04_ndvi_grid.png", dpi=130); plt.close()